# QC-C1-01: Circuit Profiling and Limited Basis-Gate Decomposition
PESU – C-DAC Quantum Hackathon 2026

**Single-file version:** upload only this notebook to Qniverse JupyterLab and run all cells. The first cell unpacks the embedded `qnvkit` package; the second reports whether the **official Qniverse QSDK** is in use.

In [ ]:
# Unpacks the embedded qnvkit package next to this notebook (pure Python, needs only numpy).
import base64, io, os, sys, zipfile
_PKG = """
...
"""
_dir = os.path.abspath("qnvkit_pkg")
zipfile.ZipFile(io.BytesIO(base64.b64decode(_PKG.replace("\n", "")))).extractall(_dir)
# Always use THIS embedded copy: put it first and drop any older qnvkit already imported or on the path.
sys.path[:] = [_dir] + [p for p in sys.path if p != _dir]
for _m in [m for m in sys.modules if m == "qnvkit" or m.startswith("qnvkit.")]:
    del sys.modules[_m]
print("qnvkit unpacked to", _dir)

In [ ]:
# Which QSDK packages does this Python environment have? (helps if the QSDK is not found below)
import sys, importlib.util, subprocess
print("Python", sys.version.split()[0])
for _n in ("Qniverse", "qniverse"):
    print("importable as %r:" % _n, importlib.util.find_spec(_n) is not None)
_pip = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True).stdout
print([l for l in _pip.splitlines() if "niverse" in l.lower() or "qsdk" in l.lower()])

In [ ]:
import qnvkit, qnvkit.qsdk_adapter as adapter          # must be imported before Qniverse
from qnvkit.metrics import profile
from qnvkit.decompose.decomposer import decompose

from Qniverse.QniverseCircuit import *
from Qniverse.gates import *
import numpy as np

print("qnvkit loaded from:", os.path.dirname(qnvkit.__file__))
print("QSDK in use:", adapter.qsdk_description())

In [ ]:
# What the real QSDK does with the constructs this demo uses. Every probe is isolated, so one
# failure does not stop the others. Please keep this output if anything below fails.
import traceback

def _probe(label, build):
    qc = QniverseCircuit()
    q = QuantumRegister(3, 'q'); c = ClassicalRegister(3, 'c')
    qc.add_register(q); qc.add_register(c)
    try:
        build(qc, q, c)
        text = adapter.read_qasm(qc)
        print("[OK]  %s\n%s\n" % (label, "\n".join("      " + l for l in text.splitlines()[4:])))
    except Exception as e:
        print("[ERR] %s -> %s: %s\n" % (label, type(e).__name__, e))

_probe("h, cp, u3, measure", lambda qc, q, c: (qc.add_gate(h, q[0]), qc.add_gate(cp(0.5), q[0], q[1]),
       qc.add_gate(u3(0.1, 0.2, 0.3), q[1]), qc.add_gate(measure, q[0], c[0])))
_probe("barrier on 2 qubits q[0],q[1]", lambda qc, q, c: (qc.add_gate(h, q[0]), qc.add_gate(barrier, q[0], q[1])))
_probe("barrier on 2 qubits q[0],q[2]", lambda qc, q, c: (qc.add_gate(h, q[0]), qc.add_gate(barrier, q[0], q[2])))
_probe("barrier on 1 qubit", lambda qc, q, c: (qc.add_gate(h, q[0]), qc.add_gate(barrier, q[0])))
_probe("reset + conditional", lambda qc, q, c: (qc.add_gate(measure, q[0], c[0]), qc.add_gate(reset, q[1]),
       qc.add_conditional_gate(c, 1, x, q[1])))
_probe("ccx, swap, crz, measure_all", lambda qc, q, c: (qc.add_gate(ccx, q[0], q[1], q[2]), qc.add_gate(swap, q[0], q[1]),
       qc.add_gate(crz(0.7), q[0], q[1]), qc.measure_all()))

# Source of the SDK's own QASM export around barriers (read-only; for debugging).
try:
    import importlib, inspect
    if not adapter.IS_LIVE_QNIVERSE:
        raise RuntimeError("the official QSDK is not in use, so there is no SDK source to show")
    _m = importlib.import_module(adapter.QSDK_IMPORT_NAME + ".transpile.qasm")
    _lines = inspect.getsource(_m).splitlines()
    _hits = [i for i, l in enumerate(_lines) if "barrier" in l.lower()]
    print("--- SDK source lines mentioning 'barrier' (with context) ---")
    _shown = set()
    for i in _hits:
        for j in range(max(0, i - 6), min(len(_lines), i + 22)):
            if j not in _shown:
                _shown.add(j); print("%4d  %s" % (j + 1, _lines[j]))
        print("      ...")
except Exception as e:
    print("source not available:", e)

## 1. Build a circuit with the QSDK

In [ ]:
qc = QniverseCircuit()
q = QuantumRegister(3, 'q')
c = ClassicalRegister(3, 'c')
qc.add_register(q)
qc.add_register(c)

qc.add_gate(h, q[0])
qc.add_gate(s, q[1])
qc.add_gate(t, q[2])
qc.add_gate(cx, q[0], q[1])
qc.add_gate(cz, q[1], q[2])
qc.add_gate(swap, q[0], q[2])
qc.add_gate(crz(0.7), q[0], q[1])
qc.add_gate(ccx, q[0], q[1], q[2])
qc.add_gate(barrier, q[0], q[2])   # QSDK barrier: start and end qubit
for i in range(3):
    qc.add_gate(measure, q[i], c[i])

original_qasm = adapter.read_qasm(qc)
print(original_qasm)

## 2. Profile
Depth uses an earliest-available-layer schedule: each operation takes one layer on every qubit it uses, operations sharing a qubit keep circuit order, operations on disjoint qubits share a layer. Measure and reset take a layer; a barrier takes none but aligns its qubits.

In [ ]:
report = profile(qc)
print(report.summary())
report.to_dict()

## 3. Decompose into a chosen basis
Returns a **new** `QniverseCircuit`; metrics before and after; equivalence checked by unitary comparison up to global phase.

In [ ]:
res = decompose(qc, {"rx", "ry", "rz", "cx"})
print(res.summary())
print(adapter.read_qasm(res.circuit))

In [ ]:
for basis in ({"u3", "cx"}, {"rz", "ry", "cz"}, {"rx", "rz", "cx"}, {"p", "ry", "cx"}):
    r = decompose(qc, basis)
    print(sorted(basis), "| gates", r.before.total_gates, "->", r.after.total_gates,
          "| depth", r.before.depth, "->", r.after.depth, "| verified:", r.verified)

## 4. Unsupported operations are reported, never emitted

In [ ]:
for basis in ({"rx", "ry", "rz"}, {"h", "t", "cx"}):
    r = decompose(qc, basis)
    print("Basis", sorted(basis), "-> circuit:", r.circuit)
    print(r.unsupported.summary(), "\n")

for bad in (set(), {"cx", "sx"}):
    try:
        decompose(qc, bad)
    except ValueError as e:
        print("Rejected", sorted(bad), ":", str(e)[:100])

## 5. Input circuit is unchanged

In [ ]:
print("Input QASM identical before/after:", adapter.read_qasm(qc) == original_qasm)
print("Output is a different object:", res.circuit is not qc)

## 6. Run original and decomposed circuits on a Qniverse simulator
Paste your Qniverse API key below (2 CPU jobs = 10 credits). The two histograms should agree within sampling noise.

In [ ]:
token = ""   # <- your Qniverse API key (keep it out of git)

if token.strip():
    import importlib, pkgutil, time
    import Qniverse
    from Qniverse.backend import Backend

    def _find(name):
        """Locates a job function wherever this QSDK version keeps it."""
        for m in pkgutil.walk_packages(Qniverse.__path__, Qniverse.__name__ + "."):
            try:
                mod = importlib.import_module(m.name)
            except Exception:
                continue
            if callable(getattr(mod, name, None)):
                print("%s found in %s" % (name, m.name))
                return getattr(mod, name)
        raise ImportError("%s not found in the QSDK; names in api: %s" % (
            name, [n for n in dir(importlib.import_module(Qniverse.__name__ + ".api")) if not n.startswith("_")]))

    run = _find("run"); fetch_job_result = _find("fetch_job_result"); job_status = _find("job_status")

    backend = Backend()
    backend.processor("CPU")
    backend.hpc("QACC_Cluster")
    backend.simulator("qasm_simulator")

    for label, circ in (("original", qc), ("decomposed", res.circuit)):
        job = run(circ, backend, 1024, token=token, job_name="c1" + label)
        t0 = time.time()
        while time.time() - t0 < 300:
            status = str(job_status(token, job)).upper()
            if "COMPLETED" in status or "FAILED" in status:
                break
            time.sleep(3)
        print(label, status, fetch_job_result(token, job))
else:
    print("No token set: skipped.")
